#### Now we evaluate the full RAG pipeline. For each generated question, we run RAG and save the answer produced by the LLM. Later, we'll compare this answer with the original FAQ answer.

This is the A->Q->A' setup:

- A = original answer in the FAQ
- Q = generated question from this answer
- A' = answer produced by our RAG system

### Data flow overview

```mermaid
flowchart LR
    A["ground_truth-new.csv<br/>question, document"] --> B["pandas DataFrame<br/>columns: question, document"]
    B --> C["records list of dicts"]

    D["FAQ data from datatalks.club"] --> E["documents: list of dicts"]
    E --> F["filter to llm-zoomcamp"]
    F --> G["build_index(documents)"]
    G --> H["minsearch index<br/>searchable by question/section/answer"]

    E --> I["doc_idx lookup table<br/>id -> full document"]

    C --> J["Evaluation loop<br/>question -> retrieve -> generate answer -> compare"]
    H --> J
    I --> J
```

Example payloads:

```python
ground_truth[0]
# {'question': 'Is it okay to join the course late if I just found it now?', 'document': '74eb249bbf'}

documents[0]
# {'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'question': '...', 'answer': '...', 'section': '...'}
```

The notebook uses two related formats:
- Ground-truth rows: `{"question": ..., "document": ...}`
- FAQ documents: `{"id": ..., "course": ..., "question": ..., "answer": ..., "section": ...}`

### Loading the data

In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [3]:
len(ground_truth)

395

In [4]:
ground_truth[0]

{'question': 'Is it okay to join the course late if I just found it now?',
 'document': '74eb249bbf'}

In [2]:
# Load the FAQ documents and the search index

from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [5]:
len(documents)

103

In [6]:
documents[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [8]:
# Create a lookup table for the original FAQ documents
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [9]:
doc_idx["74eb249bbf"]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [11]:
q = ground_truth[0]
q

{'question': 'Is it okay to join the course late if I just found it now?',
 'document': '74eb249bbf'}

In [12]:
doc_idx[q['document']]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

### Running RAG

In [13]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [14]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

In [15]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes — you can still join late if you just found the course. If you want a certificate, make sure to submit your project while submissions are still open.'

In [16]:
# Check the cost of this call

assistant.total_cost()

0.0005055

In [17]:
# Get the original answer from the document ID

doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [18]:
# Now save both answers in one record

rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Yes — you can still join late if you just found the course. If you want a certificate, make sure to submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [19]:
# Create a function that processes one ground truth record

def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [21]:
# Test it on one record

answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Yes, you can still join the course late. If you want to receive a certificate, make sure to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [22]:
# Before running the full batch, reset the usage we collected while testing:
assistant.reset_usage()

We run it for all the data, then save it in the data folder as "rag-answers.new.csv"

In [23]:
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [30]:
answers[0]

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Yes, you can still join the course late. If you want a certificate, though, you need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

## LLM as a Judge

#### A->Q->A' evaluation

We'll compare the RAG answer with the original answer from the FAQ. This checks if the RAG pipeline is producing answers that match the ground truth.



In [24]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [26]:
# Write the judge instructions. This tells the judge what to compare and how to assign the score.

aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [27]:
# Then define the prompt template. This is the data we pass to the judge for each answer

aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [29]:
rec = answers[0]
rec

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Yes, you can still join the course late. If you want a certificate, though, you need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [31]:
# Create the judge prompt:

prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

In [33]:
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

In [34]:
# Call the judge:

eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key meaning of the ground truth: late joining is allowed, but certificate eligibility requires submitting the project while submissions are open. It is semantically equivalent.', score='good')

In [35]:
# Check the cost:

calc_price(usage)

{'input_cost': 0.00022275000000000002,
 'output_cost': 0.0002295,
 'total_cost': 0.00045225}

In [36]:
# Now put the same logic into a function:

def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [37]:
# Test it on the same record

eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key meaning of the ground truth: late joining is allowed, but certificate eligibility depends on submitting the project before submissions close. It is semantically equivalent.', score='good')

### Running the judge

Run the evaluation on all answers and save it

In [38]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [41]:
# load the saved data

import pandas as pd

df_eval = pd.read_csv("data/rag-evaluations-new.csv")

In [42]:
df_eval

,question,document,score,reasoning
0,Is it okay to join the course late if I just f...,74eb249bbf,good,The AI answer preserves the ground truth meani...
1,Can I still take this course even if I missed ...,74eb249bbf,good,The AI answer preserves the core meaning: you ...
2,If I join after the course has already started...,74eb249bbf,good,The AI answer preserves the key point: joining...
3,Do I need to submit my project before submissi...,74eb249bbf,good,The AI answer preserves the key point of the g...
4,I’m a bit late to the course—what do I need to...,74eb249bbf,good,The AI answer includes the core requirement fr...
...,...,...,...,...
390,Why do I get a 401 Client Error when using the...,4b30b918bc,good,The ground truth says a 401 Client Error may i...
391,What's the easiest way to force-install reques...,4b30b918bc,good,The AI answer gives the exact installation com...
392,Can I install requests straight from the GitHu...,4b30b918bc,good,The AI answer conveys the same core instructio...
393,"If pip keeps pulling requests v2.28, what exac...",4b30b918bc,good,The AI answer gives the exact install command ...


In [43]:
df_eval.score.value_counts()

score
good    379
bad      16
Name: count, dtype: int64

In [45]:
df_eval.score.value_counts(normalize=True)

score
good    0.959494
bad     0.040506
Name: proportion, dtype: float64

In [46]:
# Check the results

good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 379/395 = 95.95%


In [47]:
# Look at the "bad" cases to understand what went wrong:

df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
15,How do the free GPU hours work on these cloud ...,c6c2888275,bad,The AI answer does not convey the ground truth...
29,Is peer-review of the capstone project require...,69d122f12e,bad,The AI answer is not semantically equivalent t...
38,How will I know when a module is actually read...,96286b4be4,bad,The AI answer does not convey the ground truth...
73,Which model should I use in chat.completions.c...,152af39a53,bad,The ground truth says the issue is insufficien...
106,Do I need an OpenAI API key just to check how ...,fe8fed31e6,bad,The AI answer does not address the question or...
